In [ ]:
#from Processor import get_dataset#, get_temprel_counts
from Reader import TBDenseReader, TempEval3Reader, MAVENReader, OzRockReader, TweetsReader, WikiWarsReader
from copy import deepcopy
def get_temprel_props(data, type="per_sample"):
    total = {}
    if type == "per_sample":
        for split in data:
            split_counts = []
            for sample in data[split]:
                counts = {}
                for temprel in sample["ee_temprels"]:
                    if temprel["rel"] in ["BEFORE", "AFTER", "CONTAINS"]:
                        counts["BIG"] = counts.get("BIG", 0) + 1
                    else:
                        counts["SMALL"] = counts.get("SMALL", 0) + 1
                split_counts.append(counts)
            total[split] = split_counts
    else:
        for split in data:
            for sample in data[split]:
                for temprel in sample["ee_temprels"]:
                    if temprel["rel"] in ["BEFORE", "AFTER", "CONTAINS"]:
                        total["BIG"] = total.get("BIG", 0) + 1
                    else:
                        total["SMALL"] = total.get("SMALL", 0) + 1
    return total

def data_temprel_select(data):
    counts = {"BEFORE": 0, "AFTER": 0, "DURING": 0, "CONTAINS": 0, "OVERLAPS": 0, "EQUALS": 0, "IDENTITY": 0}
    target = {"BEFORE": 75855, "AFTER": 75855, "DURING": 75855, "CONTAINS": 75855, "OVERLAPS": 75855, "EQUALS": 75855, "IDENTITY": 75855}
    balanced = {}

    for name in data:
        balanced[name] = {}
        for split in data[name]:
            balanced[name][split] = []
            for sample in data[name][split]:
                new_sample = sample.copy()
                new_sample["ee_temprels"] = []
                for temprel in sample["ee_temprels"]:

                    if name in ["TempEval3", "TBDense"] and split == "train" and temprel['rel'] == "BEFORE":
                        continue

                    elif temprel['rel'] == "BEFORE" and counts["BEFORE"] == target["BEFORE"] and counts["AFTER"] < target["AFTER"]:
                        counts["AFTER"] += 1
                        new_sample["ee_temprels"].append({"rel": "AFTER", "e1": temprel["e2"], "e2": temprel["e1"]})

                    elif counts[temprel['rel']] < target[temprel['rel']]:
                            counts[temprel['rel']] += 1
                            new_sample["ee_temprels"].append(temprel)

                balanced[name][split].append(new_sample)

    return balanced, counts

In [4]:
def reindex(data):
    eid2index = {}
    tid2index = {}
    for inst in data["instances"]:
        if inst["type"] == "EVENT":
            eid2index[inst["id"]] = len(eid2index)
            inst['id'] = eid2index[inst['id']]
        else:
            tid2index[inst["id"]] = len(tid2index)
            inst['id'] = tid2index[inst['id']]

    for temprel in data["ee_temprels"]:
        temprel['e1'] = eid2index[temprel['e1']]
        temprel['e2'] = eid2index[temprel['e2']]

    for eventtimes in data["event_times"]:
        if eventtimes['time'][0] == "e":
            time = eventtimes['event']
            event = eventtimes['time']
            eventtimes['event'] = event
            eventtimes['time'] = time

        eventtimes['event'] = eid2index[eventtimes['event']]
        eventtimes['time'] = tid2index[eventtimes['time']]

    return data

def get_dataset(reader):
    data = reader.read()
    if type(reader) is MAVENReader:
        data.pop("test")
    for split in data:
        for sample in data[split]:
            sample = reindex(sample)
    return data

In [ ]:
data = {}
rawdata_path = "D:\\GeoTKG\\rawdata\\"
for name, reader in [("TempEval3", TempEval3Reader),("TBDense", TBDenseReader),('MAVEN_ERE', MAVENReader),]:
    data[name] = get_dataset(reader(rawdata_path + name))

In [18]:
dc = deepcopy(data)
bal, cnts = data_temprel_select(dc)

In [19]:
cnts

{'BEFORE': 75855,
 'AFTER': 75855,
 'DURING': 3451,
 'CONTAINS': 75855,
 'OVERLAPS': 4191,
 'EQUALS': 18302,
 'IDENTITY': 39734}

In [35]:
counts = {}
for name in ["TBDense", "TempEval3", "MAVEN_ERE"]:
    counts[name] = get_temprel_props(bal[name], type="whole")

In [36]:
counts

{'TBDense': {'BIG': 2071, 'SMALL': 662},
 'TempEval3': {'BIG': 38385, 'SMALL': 14578},
 'MAVEN_ERE': {'BIG': 187109, 'SMALL': 50438}}

In [ ]:
for k in counts:
    print(f'{k}: {counts[k]["BIG"] + counts[k]["SMALL"]}')

TBDense: 2733
TempEval3: 52963
MAVEN_ERE: 237547


In [39]:
for dset in dc:
    for splt in dc[dset]:
        print(f'{dset} - {splt}: {len(bal[dset][splt]) == len(dc[dset][splt])}')

TempEval3 - eval: True
TempEval3 - test: True
TempEval3 - train: True
TBDense - eval: True
TBDense - test: True
TBDense - train: True
MAVEN_ERE - eval: True
MAVEN_ERE - train: True
